[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/rare_variant.ipynb)

# SAIGE-gene analysis
Let's run a SAIGE-gene analysis! First, we'll run some toy data through the SAIGE-gene software so that you're able to run it in the future.

## SAIGE setup
First though, we need to go through the faff of getting SAIGE installed.

Run the cells below to initialize the multi-language environment (Python, Bash, R) and pull the SAIGE Docker container, and then let's get SAIGE installed and ready for a collection of analyses.

In [1]:
# 1. Load R magic extension
%load_ext rpy2.ipython

In [2]:
%%bash
# Install SAIGE - we can follow the pixi installation from the SAIGE website
src_branch=main
repo_src_url=https://github.com/saigegit/SAIGE
git clone --depth 1 -b $src_branch $repo_src_url

Cloning into 'SAIGE'...


In [17]:
# 1. Install pixi
!curl -fsSL https://pixi.sh/install.sh | bash

import os
os.environ['PATH'] = f"{os.environ['HOME']}/.pixi/bin:{os.environ['PATH']}"
%cd /content/SAIGE
!pixi install
!rm -rf ~/.cache
!pixi run Rscript -e 'install.packages("lintools", repos="https://cloud.r-project.org")'

This script will automatically download and install Pixi (latest) for you.
Getting it from this url: https://github.com/prefix-dev/pixi/releases/latest/download/pixi-x86_64-unknown-linux-musl.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 31.3M  100 31.3M    0     0  52.2M      0 --:--:-- --:--:-- --:--:-- 52.2M
The 'pixi' binary is installed into '/root/.pixi/bin'
/content/SAIGE
✔ The default environment has been installed.
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/lintools_0.1.7.tar.gz'
Content type 'application/x-gzip' length 82939 bytes (80 KB)
downloaded 80 KB

* installing *source* package ‘lintools’ ...
** this is

In [19]:
# Make sure you are in the SAIGE directory
%cd /content/SAIGE

# 1. Download and extract the specific PLINK2 release
!curl -L https://github.com/chrchang/plink-ng/archive/refs/tags/v2.0.0-a.6.16.tar.gz | tar -zx

# (Optional safety step: remove any empty plink-ng folder Git might have made)
!rm -rf plink-ng

# 2. Rename the folder so SAIGE can find it
!mv plink-ng-2.0.0-a.6.16 plink-ng

# 3. Compile the PLINK2 library using the Pixi compiler
!pixi run x86_64-conda-linux-gnu-cc -std=c++14 -fPIC -O3 -I.pixi/envs/default/include -L.pixi/envs/default/lib -o libplink2_includes.a plink-ng/2.0/include/*.cc -shared -lz -lzstd -lpthread -lm -ldeflate

# 4. Move the compiled library into the Pixi environment
!mv libplink2_includes.a .pixi/envs/default/lib

/content/SAIGE
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 4607k    0 4607k    0     0  2853k      0 --:--:--  0:00:01 --:--:-- 4808k


In [23]:
import os

# 1. Nuke Colab's system R paths so they stop poisoning the Pixi environment
for var in ['R_HOME', 'R_LIBS', 'R_LIBS_USER', 'R_LIBS_SITE']:
    if var in os.environ:
        del os.environ[var]

# 2. Force R to exclusively read from and install to the isolated Pixi library
os.environ['R_LIBS_USER'] = '/content/SAIGE/.pixi/envs/default/lib/R/library'

# 3. Change directory and finish the installation (C++ won't recompile, so it will be fast)
%cd /content/SAIGE
!pixi run R CMD INSTALL .

/content/SAIGE
* installing to library ‘/content/SAIGE/.pixi/envs/default/lib/R/library’
* installing *source* package ‘SAIGE’ ...
** this is package ‘SAIGE’ version ‘1.5.1’
** using staged installation
** libs
using C++ compiler: ‘x86_64-conda-linux-gnu-c++ (conda-forge gcc 15.2.0-19) 15.2.0’
using C++14
make: Nothing to be done for 'all'.
installing to /content/SAIGE/.pixi/envs/default/lib/R/library/00LOCK-SAIGE/00new/SAIGE/libs
** R
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (SAIGE)


The above cell should end with `* DONE (SAIGE)`. If it doesn't, shout - you won't be the only one, and we'll help you diagnose what went wrong.

If it does....read on!

# Step 0: creating a sparse GRM

*Note: This step is only needed for region- and gene-based tests (SAIGE-gene). The sparse GRM only needs to be created once for each data set, e.g. a biobank, and can be used for all different phenotypes as long as all tested samples are included in it.*

In [31]:
%cd /content/SAIGE
!pixi run Rscript extdata/createSparseGRM.R --help

Warning messages:
1: replacing previous import ‘data.table::first’ by ‘dplyr::first’ when loading ‘SAIGE’ 
2: replacing previous import ‘data.table::between’ by ‘dplyr::between’ when loading ‘SAIGE’ 
3: replacing previous import ‘data.table::last’ by ‘dplyr::last’ when loading ‘SAIGE’ 
Loading required package: optparse
Warning message:
package ‘optparse’ was built under R version 4.5.3 
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /content/SAIGE/.pixi/envs/default/lib/libopenblasp-r0.3.33.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Etc/UTC
tzcod

About to run

extdata/createSparseGRM.R \
--plinkFile=extdata/input/nfam_100_nindep_0_step1_includeMoreRareVariants_poly \
--nThreads=4 \
--outputPrefix=extdata/output/sparseGRM \
--numRandomMarkerforSparseKin=1000 \
--relatednessCutoff=0.125

Take a look through the options above, and think about what each of these options are doing.

To summarise, there's a series of input files, and a bunch of output files get created. We also provide a couple of options to determine a relatedness cutoff for the GRM, and how many concurrent threads to run.

Input files
Genotype file for constructing the genetic relationship matrix SAIGE takes the PLINK binary file for the genotypes and assumes the file prefix is the same one for .bed, .bim. and .fam. So that means that the inputs SAIGE is taking are

./input/nfam_100_nindep_0_step1_includeMoreRareVariants_poly.v2.bed ./input/nfam_100_nindep_0_step1_includeMoreRareVariants_poly.v2.bim ./input/nfam_100_nindep_0_step1_includeMoreRareVariants_poly.v2.fam

Output files
It's important to remember the settings you used for an analysis run. Luckily, SAIGE does this for you! Output files are named using the output prefix (--outputPrefix), but include options that were to the run in the filename. Nice.

The sparse GRM file ./output/sparseGRM_relatednessCutoff_0.125_1000_randomMarkersUsed.sparseGRM.mtx
This file can be opened using the readMM function in the R library Matrix
The sample IDs for the rows in the sparse GRM file ./output/sparseGRM_relatednessCutoff_0.125_1000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt


In [34]:
!pixi run Rscript extdata/createSparseGRM.R \
--plinkFile=extdata/input/nfam_100_nindep_0_step1_includeMoreRareVariants_poly \
--nThreads=4 \
--outputPrefix=extdata/output/sparseGRM \
--numRandomMarkerforSparseKin=1000 \
--relatednessCutoff=0.125

Streaming output truncated to the last 5000 lines.
  [11,] .         .         .         .         .         .         .        
  [12,] .         .         .         .         .         .         .        
  [13,] .         .         .         .         .         .         .        
  [14,] .         .         .         .         .         .         .        
  [15,] .         .         .         .         .         .         .        
  [16,] .         .         .         .         .         .         .        
  [17,] .         .         .         .         .         .         .        
  [18,] .         .         .         .         .         .         .        
  [19,] .         .         .         .         .         .         .        
  [20,] .         .         .         .         .         .         .        
  [21,] .         .         .         .         .         .         .        
  [22,] .         .         .         .         .         .         .        
  [23,] .    

The screen output ends with the following text if the job above has been run successfully

If you see this, move to the next step!

Step 1: fitting the null logistic/linear mixed model

Let's fit a null logistic mixed model by running step 1 in SAIGE. You did this the other day, for single variant association testing. Note that it's the same procedure, because the null model is the same (in the sense that there's no genetic association).

Depending on whether y is continuous or binary, SAIGE will fit a linear or logistic mixed model under the null hypothesis of no genetic associaiton. For example, in the case of binary y we might have

logit( π ) = X1 + X2 + b
π​ : The probability that the binary outcome y = 1.
y is the phenotype with 0/1
X1 and X2 are the covariates, e.g. age, sex, genetic PCs
b ~ N(0, tau * GRM) and used to account for the sample relatedness through GRM (it's a correlated noise term)

Run the following command to fit the null logistic mixed model

bash step1_binary.sh

Take a look under the hood to see what this script is doing

less -S step1_binary.sh

This should be familiar! It's very similar to the null fit from Wei's session earlier in the week.

As a reminder, you can see all of the options for SAIGE step 1 in gory detail by running

step1_fitNULLGLMM.R --help

Take a look in step1_fitNULLGLMM.log

cat step1_fitNULLGLMM.log

Does the log file end with...

If you see this, move to the next step!

Step 2: performing the set-based association tests

Input files

Dosage/genotype file containing dosages or genotypes of markers to test
SAIGE supports different formats for dosages: VCF, BCF, BGEN and SAV. We'll use BGEN in the example today
./input/genotype_100markers.bgen
./input/genotype_100markers.bgen.bgi

Sample file
This file contains one column for sample IDs corresponding to the sample order in the dosage file. No header is included. The file is ONLY for BGEN input.
./input/samplelist.txt

Model file from step 1
./output/example_binary_sparseGRM.rda

Variance ratio file from step 1, but has categorical variance ratios ./output/example_binary_sparseGRM.varianceRatio.txt

Group file (specific to SAIGE-gene)
./input/group_new_chrposa1a2.txt

Sparse GRM (output by step 0, same input for step 1) ./output/sparseGRM_relatednessCutoff_0.125_1000_randomMarkersUsed.sparseGRM.mtx

Sparse GRM sample IDs (output by step 0, same input for step 1) ./output/sparseGRM_relatednessCutoff_0.125_1000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt

Take a look at the group file by running
cat -S ./input/group_new_chrposa1a2.txt

The group file contains at least two lines for each gene/set. The first line has the gene/set name, followed by "var" and variant ids. The second line contains the gene/set name, followed by "anno" and variant annotations. There can be a third line, which has the gene/set name, followed by "weight" and user-specified weights for variants. The third line is optional. If it is not included in the group file, the weights will be calcuated as beta(MAF, 1, 25) by default.

Run the following command to perform the set-based association tests

bash step2_test.sh

The screen output ends with the following text if the job above has been run successfully


Congratulations, you've run SAIGE-gene from start to finish!

Take a look under the hood to see what this last script you ran is doing

cat step2_test.sh

Wow, there are a lot of options passed.

Try running step2_SPAtests.R --help to have a read about what each of them are doing.


SAIGE-gene analysis, Part 2: Real data.

With real data, it takes a little while to generate the results. To save time, in the SAIGE/ukbiobank folder, you'll find an example of some results of applying SAIGE-gene to a trait in the UK Biobank.

Before you start , take a look at the SAIGE Readme to get a sense of what the output files should contain, and what the columns of the output file are.

You can look inside the file by moving to the directory, and using the command zless. For this part, make sure you're in the 'Terminal' tab.

cd ~/practicals/5.2.RareVariantAssociation_DuncanPalmer/final/SAIGE/ukbiobank
zless -S pilot-traits_uk-biobank_gene_uk-biobank.palmer.pilot.AFib.JULY23Freeze.ALL.EUR.28671.373704.SAIGE.gene.20240110.txt.gz

Question 2.1. What classes of tests have been run? (select all that apply) Hint: the Pvalue column (which might seem slightly ambiguous(!), comes from a SKAT-O test)

If you said four different annotations, you're right!
pLoF
damaging_missense_or_protein_altering
other_missense_or_protein_altering
synonymous

In the call to SAIGE-gene

pLoF;damaging_missense_or_protein_altering and pLoF;damaging_missense_or_protein_altering;other_missense_or_protein_altering;synonymous

were also passed, which combine those annotations together.

These are included in the Group column of the output.


Question 2.3. What MAF cutoffs were used? (select all that apply)

You should have said 0.01, 0.001, 0.0001.

This information is present in the max_MAF.

Cauchy combination tests are then applied across these (mask, MAF) pairs for each class of test (Burden, SKAT, SKAT-O).

Question 2.4. How do the results in synonymous variation look? Can you create a QQ plot?
Here's some code to generate a QQ plot, this time with added ggplot2...

library(data.table)
library(dplyr)
library(tidyr)
library(ggplot2)

setwd("~/practicals/5.2.RareVariantAssociation_DuncanPalmer/final/SAIGE")
dt <- fread("ukbiobank/pilot-traits_uk-biobank_gene_uk-biobank.palmer.pilot.AFib.JULY23Freeze.ALL.EUR.28671.373704.SAIGE.gene.20240110.txt.gz")
dt <- dt %>% filter(Group != "Cauchy")
dt <- dt %>%
  pivot_longer(cols = c(Pvalue, Pvalue_Burden, Pvalue_SKAT),
               names_to = "class", values_to = "Pvalue") %>%
  mutate(
    class = recode(class,
                   "Pvalue" = "SKAT-O",
                   "Pvalue_Burden" = "Burden",
                   "Pvalue_SKAT" = "SKAT"),
    Group = recode(Group,
                   "damaging_missense_or_protein_altering" = "Damaging missense",
                   "other_missense_or_protein_altering" = "Other missense",
                   "pLoF" = "pLoF",
                   "pLoF;damaging_missense_or_protein_altering" = "pLoF, damaging missense",
                   "pLoF;damaging_missense_or_protein_altering;other_missense_or_protein_altering;synonymous" = "pLoF, damaging missense,\nother missense, synonymous",
                   "synonymous" = "Synonymous",
    )
  ) %>%
  group_by(class, Group, max_MAF) %>%
  setorder("Pvalue") %>%
  mutate(Pvalue_exp = seq(1,n())/(n()+1))

dt <- dt %>% filter(Group == "Synonymous")

ggplot(dt %>% setorder("class"), aes(x=-log10(Pvalue_exp), y=-log10(Pvalue), col=class)) +
  geom_point() + facet_wrap(~max_MAF+Group, scales = 'free') +
  theme_bw() + geom_abline(intercept=0,slope=1, col='grey')

Discuss in your group.

You should have noticed that there's lift off from y=x for synonymous variation, suggesting that there could well be common variant associations nearby that you're tagging. It might be a good idea to look and see if there are common variant associations close to those lead gene-phenotype associations.

Thank you for working through this tutorial!